In [5]:
from finvizfinance.earnings import Earnings
import pandas as pd
from datetime import datetime

ONE_BILLION = 1_000_000_000

# -----------------------
# FETCH THIS MONTH
# -----------------------
earnings = Earnings()
earnings._set_period("This Month")

df = earnings.df.copy()

if df.empty:
    raise RuntimeError("No earnings data returned from Finviz")

# -----------------------
# PARSING HELPERS
# -----------------------
def parse_market_cap(value):
    if pd.isna(value):
        return 0
    value = str(value).upper().strip()
    multipliers = {"T": 1e12, "B": 1e9, "M": 1e6, "K": 1e3}
    try:
        return float(value[:-1]) * multipliers.get(value[-1], 1)
    except:
        return 0

def parse_earnings_date(value):
    if pd.isna(value):
        return None
    clean = value.replace("/a", "").replace("/b", "").strip()
    try:
        return datetime.strptime(
            f"{clean} {datetime.today().year}",
            "%b %d %Y"
        )
    except:
        return None

def parse_earnings_timing(value):
    if pd.isna(value):
        return "Unknown"
    value = value.lower()
    if value.endswith("/b"):
        return "Before Market Open"
    if value.endswith("/a"):
        return "After Market Close"
    return "Unknown"

# -----------------------
# ENRICH DATA
# -----------------------
df["Market Cap Numeric"] = df["Market Cap"].apply(parse_market_cap)
df["Earnings Date"] = df["Earnings"].apply(parse_earnings_date)
df["Earnings Timing"] = df["Earnings"].apply(parse_earnings_timing)

# -----------------------
# FILTER + RANK
# -----------------------
monthly_df = (
    df[df["Market Cap Numeric"] >= ONE_BILLION]
    .dropna(subset=["Earnings Date"])
    .sort_values(["Earnings Date", "Market Cap Numeric"], ascending=[True, False])
    .groupby("Earnings Date")
    .head(10)
    .reset_index(drop=True)
)

# -----------------------
# FINAL COLUMNS
# -----------------------
monthly_df = monthly_df[
    ["Earnings Date", "Ticker", "Market Cap", "Market Cap Numeric", "Earnings Timing"]
]

monthly_df


,Earnings Date,Ticker,Market Cap,Market Cap Numeric,Earnings Timing
0,2026-02-02,PLTR,3.763000e+11,3.763000e+11,After Market Close
1,2026-02-02,DIS,1.848200e+11,1.848200e+11,Before Market Open
2,2026-02-02,MFG,1.105300e+11,1.105300e+11,After Market Close
3,2026-02-02,SPG,6.184000e+10,6.184000e+10,After Market Close
4,2026-02-02,NXPI,5.553000e+10,5.553000e+10,After Market Close
...,...,...,...,...,...
179,2026-02-27,HE,2.740000e+09,2.740000e+09,After Market Close
180,2026-02-27,NWN,1.980000e+09,1.980000e+09,Before Market Open
181,2026-02-27,DK,1.850000e+09,1.850000e+09,Before Market Open
182,2026-02-27,SHO,1.670000e+09,1.670000e+09,Before Market Open


In [6]:
with pd.ExcelWriter("earnings_this_month.xlsx") as writer:
    monthly_df.to_excel(writer, sheet_name="Earnings Calendar", index=False)

"earnings_this_month.xlsx"


'earnings_this_month.xlsx'

In [7]:
import yfinance as yf

NUM_PREVIOUS_EARNINGS = 8

def earnings_reactions_df(tickers, num_earnings=6):
    all_rows = []

    for ticker in tickers:
        print(f"Fetching earnings reactions for {ticker}...")
        t = yf.Ticker(ticker)

        try:
            earnings_df = t.get_earnings_dates(limit=num_earnings)
            if earnings_df is None or earnings_df.empty:
                continue

            if isinstance(earnings_df.index, pd.DatetimeIndex):
                earnings_df = earnings_df.reset_index().rename(columns={"index": "Earnings Date"})

            if "Earnings Date" not in earnings_df.columns:
                earnings_df = earnings_df.rename(columns={earnings_df.columns[0]: "Earnings Date"})

            earnings_df = earnings_df.head(num_earnings)
        except Exception as e:
            print(f"  ❌ Failed for {ticker}: {e}")
            continue

        for _, row in earnings_df.iterrows():
            edate = pd.to_datetime(row["Earnings Date"]).date()
            start = edate - pd.Timedelta(days=1)
            end = edate + pd.Timedelta(days=2)

            hist = t.history(start=start, end=end, auto_adjust=False)
            if hist.empty:
                continue

            hist.index = hist.index.date

            day_before = hist.get(edate - pd.Timedelta(days=1))
            day_of = hist.get(edate)
            day_after = hist.get(edate + pd.Timedelta(days=1))

            close_before = hist.loc[hist.index == edate - pd.Timedelta(days=1), "Close"].iloc[0] if (edate - pd.Timedelta(days=1)) in hist.index else None
            close_day = hist.loc[hist.index == edate, "Close"].iloc[0] if edate in hist.index else None
            close_after = hist.loc[hist.index == edate + pd.Timedelta(days=1), "Close"].iloc[0] if (edate + pd.Timedelta(days=1)) in hist.index else None

            def pct(a, b):
                return round((b - a) / a * 100, 2) if a and b else None

            all_rows.append({
                "Ticker": ticker,
                "Earnings Date": edate,
                "Close Before": close_before,
                "Close DayOf": close_day,
                "Close After": close_after,
                "% Before→DayOf": pct(close_before, close_day),
                "% DayOf→After": pct(close_day, close_after),
                "% Before→After": pct(close_before, close_after),
            })

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df["Earnings Date"] = pd.to_datetime(df["Earnings Date"])
        df = df.sort_values(["Ticker", "Earnings Date"], ascending=[True, False])

    return df


In [8]:
def gamble_summary(df):
    if df.empty:
        return pd.DataFrame()

    g = df.dropna(subset=["% Before→After"]).groupby("Ticker")

    summary = g["% Before→After"].agg(
        num_earnings="count",
        avg_before_after="mean",
        median_before_after="median"
    ).reset_index()

    summary["avg_abs_move"] = g["% Before→After"].apply(lambda x: x.abs().mean()).values
    summary["pct_pos"] = g["% Before→After"].apply(lambda x: (x > 0).mean() * 100).values
    summary["pct_neg"] = 100 - summary["pct_pos"]

    summary["gamble_flag"] = summary.apply(
        lambda r: "Yes" if (r["avg_abs_move"] >= 5 and r["pct_pos"] >= 60) else "No",
        axis=1
    )

    return summary.round(2)


In [9]:
tickers = monthly_df["Ticker"].unique().tolist()

earnings_df = earnings_reactions_df(tickers, NUM_PREVIOUS_EARNINGS)
gamble_df = gamble_summary(earnings_df)

earnings_df, gamble_df


Fetching earnings reactions for PLTR...


$PLTR: possibly delisted; no price data found  (1d 2026-05-03 -> 2026-05-06) (Yahoo error = "Data doesn't exist for startDate = 1777780800, endDate = 1778040000")


Fetching earnings reactions for DIS...


$DIS: possibly delisted; no price data found  (1d 2026-05-05 -> 2026-05-08) (Yahoo error = "Data doesn't exist for startDate = 1777953600, endDate = 1778212800")


Fetching earnings reactions for MFG...


...

Fetching earnings reactions for SHO...


$SHO: possibly delisted; no price data found  (1d 2026-02-26 -> 2026-03-01) (Yahoo error = "Data doesn't exist for startDate = 1772082000, endDate = 1772341200")


Fetching earnings reactions for VIA...


$VIA: possibly delisted; no price data found  (1d 2026-02-26 -> 2026-03-01) (Yahoo error = "Data doesn't exist for startDate = 1772082000, endDate = 1772341200")


(    Ticker Earnings Date  Close Before  Close DayOf  Close After  \
 713    AAP    2025-10-30     55.130001    50.700001    47.130001   
 714    AAP    2025-08-14     61.810001    56.849998    56.840000   
 715    AAP    2025-05-22     31.309999    49.169998    48.669998   
 716    AAP    2025-02-26     45.880001    37.700001    36.959999   
 717    AAP    2024-11-14     40.939999    41.200001    37.689999   
 ..     ...           ...           ...          ...          ...   
 645    ZTS    2025-05-06    158.059998   149.869995   155.990005   
 646    ZTS    2025-02-13    173.880005   164.929993   157.520004   
 647    ZTS    2024-11-04           NaN   175.179993   175.270004   
 648    ZTS    2024-08-06    174.820007   185.289993   184.770004   
 649    ZTS    2024-05-02    158.500000   167.229996   167.070007   
 
      % Before→DayOf  % DayOf→After  % Before→After  
 713           -8.04          -7.04          -14.51  
 714           -8.02          -0.02           -8.04  
 715    

In [10]:
with pd.ExcelWriter(
    "earnings_this_month.xlsx",
    mode="a",
    engine="openpyxl",
    if_sheet_exists="replace"
) as writer:

    monthly_df.to_excel(writer, sheet_name="Earnings Calendar", index=False)
    earnings_df.to_excel(writer, sheet_name="Earnings Reactions", index=False)
    gamble_df.to_excel(writer, sheet_name="Gamble Summary", index=False)

"earnings_this_month.xlsx updated"


'earnings_this_month.xlsx updated'

In [15]:
import pandas as pd
import numpy as np
import json
import os

# -----------------------
# Helper: make DataFrame JSON-safe
# -----------------------
def df_to_json_safe(df: pd.DataFrame) -> list[dict]:
    df = df.copy()
    # Convert datetime columns to YYYY-MM-DD
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].dt.strftime("%Y-%m-%d")
    # Replace NaN / NaT
    df = df.replace({np.nan: None})
    return df.to_dict(orient="records")

# -----------------------
# Load Excel
# -----------------------
excel_file = "earnings_this_month.xlsx"  # adjust path if needed
xls = pd.ExcelFile(excel_file)

print("Sheets found:", xls.sheet_names)

# -----------------------
# Make output folder
# -----------------------
output_dir = "json_exports"
os.makedirs(output_dir, exist_ok=True)

# -----------------------
# Loop through sheets and save each as a JSON file
# -----------------------
for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet)
    json_data = df_to_json_safe(df)
    output_path = os.path.join(output_dir, f"{sheet.replace(' ', '_').lower()}.json")
    with open(output_path, "w") as f:
        json.dump(json_data, f, indent=2)
    print(f"✅ {sheet} → {output_path}")


Sheets found: ['Earnings Calendar', 'Earnings Reactions', 'Gamble Summary']
✅ Earnings Calendar → json_exports/earnings_calendar.json
✅ Earnings Reactions → json_exports/earnings_reactions.json
✅ Gamble Summary → json_exports/gamble_summary.json
